# LLM 처리 결과를 서버 DB로 (화면이 읽는 곳)

`news_extract` → `news_track` → 요약 노트북 4개가 `source/live/`에 쓴 결과 파일을 서버 DB의 `news` 영역에 올린다.
고객 화면(web)은 이 표만 읽는다. 파일은 그대로 두고, 몇 번을 다시 돌려도 같은 결과가 되게 덮어쓴다.

| 표 | 파일 | 화면에서 |
|---|---|---|
| news.llm_articles | articles.jsonl(기사별 추출) | 뉴스 카드 제목을 한국어 한 줄 요약으로, 기사 종류·업종, 언론사 원문 주소(`original_url`, '원문 보기'를 구글 뉴스 대신 언론사로 바로) |
| news.stories | stories.csv(같은 사건 묶음) + llm_source.jsonl의 news_id(같은 소식) | 같은 사건 기사 묶기, 뉴스 상세 '같은 소식, 다른 매체'·후속 보도 |
| news.importance | importance.csv(소식별 소식·사건 보도 매체 수) | 중요한 사건 먼저 |
| news.hot_macro | macro_hot_issues.jsonl + macro_headlines.jsonl(제목·소제목) | 뉴스 브리핑 시장 · AI 전체 요약, 홈·3분 브리핑 |
| news.hot_sector | sector_hot_issues.jsonl | 산업별 · 업종 한 줄 |
| news.hot_company | company_hot_issues.jsonl | 내 종목 · 종목 한 줄과 긍정·부정 |
| news.company_summary | company_summaries.jsonl | 내 종목 · 종목 요약 |
| news.company_relation | company_relations.jsonl(news_company_relation) | 종목–해외 상대 관계 한 문장(짝마다 마지막 줄) |
| news.company_reason | company_reasons.jsonl(news_company_reason) | 종목마다 밤사이 기사 한 문장 · 관계사 기사면 관계 문장(날짜·종목마다 마지막 줄) |
| news.stock_impacts | articles.jsonl(기사별 회사 판정) + 수집 DB hits + news.term_links + relation_direction_rules.csv | 기사 × 내 종목 긍정·부정(밤사이 종목 줄, 뉴스 블록·한눈에 카드, 기사 줄) |
| news.article_ko | ko_summaries.jsonl(news_ko_summary) | 뉴스 상세 · AI 요약(한국어 줄글 번역), 영향 한 문장 |

요약 파일은 같은 날짜로 여러 번 쌓이므로(실시간 처리) 날짜마다(종목은 날짜·종목마다) 가장 늦게 만든 것만 올린다.
`news` 영역은 서버 관리자 계정이 한 번 만들어 두었다(수집 계정 gnews 소유, 화면용 읽기 계정 gnews_read·gnews_app 읽기).

In [ ]:
import csv, json, os
from pathlib import Path

import pandas as pd
import sqlalchemy as sa
from sqlalchemy.dialects.postgresql import JSONB, insert as pg_insert

ROOT = Path.cwd()                                     # news_search
PROJ = next((p for p in [ROOT, *ROOT.parents] if (p / '.env').exists()), ROOT.parent)
LIVE = ROOT / 'source/live'
DB_ENV = 'GNEWS_DB_URL'                               # 수집 계정(news 영역 쓰기)
SYNC_DAYS = 2                                         # 기사 추출 결과는 처음 수집한 지 이 날수 안의 것만 다시 올린다(None이면 전부). 요약은 날짜별 최신 것만이라 늘 전부


def read_env(path=PROJ / '.env'):
    env = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env


engine = sa.create_engine(os.environ.get(DB_ENV) or read_env()[DB_ENV], pool_pre_ping=True)
print('DB:', engine.url.render_as_string(hide_password=True), '/ 결과 폴더', LIVE)

## 1. 표
기사 표의 `impacts`는 추출 결과의 회사별 영향(회사 이름·영향)을, `extraction`은 추출 결과 전체를 그대로 담는다.

In [ ]:
meta = sa.MetaData(schema='news')
TS = sa.DateTime(timezone=True)
T_ART = sa.Table('llm_articles', meta,
    sa.Column('key', sa.Text, primary_key=True), sa.Column('report', sa.Text), sa.Column('status', sa.Text),
    sa.Column('short_summary', sa.Text), sa.Column('news_kind', sa.Text), sa.Column('article_types', JSONB),
    sa.Column('sectors', JSONB), sa.Column('impacts', JSONB), sa.Column('extraction', JSONB), sa.Column('extract_error', sa.Text),
    sa.Column('published_utc', TS), sa.Column('original_url', sa.Text), sa.Column('updated_at', TS, server_default=sa.func.now()))
T_STORY = sa.Table('stories', meta, sa.Column('key', sa.Text, primary_key=True), sa.Column('story_id', sa.Text), sa.Column('joined_by', sa.Text),
    sa.Column('news_id', sa.Text))   # 사건 안의 같은 소식 번호(2026-11-22 더함)
T_IMP = sa.Table('importance', meta, sa.Column('story_id', sa.Text, primary_key=True), sa.Column('news_id', sa.Text, primary_key=True), sa.Column('short_summary', sa.Text),   # 소식(news_id)마다 한 줄
    sa.Column('news_outlets', sa.Integer), sa.Column('story_outlets', sa.Integer), sa.Column('story_articles', sa.Integer),
    sa.Column('first_published', sa.Text), sa.Column('story_last_published', sa.Text))
T_MACRO = sa.Table('hot_macro', meta, sa.Column('date', sa.Text, primary_key=True), sa.Column('created_kst', sa.Text),
    sa.Column('lines', JSONB), sa.Column('keys', JSONB), sa.Column('payload', JSONB),
    sa.Column('title', sa.Text), sa.Column('headings', JSONB), sa.Column('keywords', JSONB))   # 제목·소제목·키워드는 news_headline.ipynb
T_SECTOR = sa.Table('hot_sector', meta, sa.Column('date', sa.Text, primary_key=True), sa.Column('sector', sa.Text, primary_key=True),
    sa.Column('created_kst', sa.Text), sa.Column('line', sa.Text), sa.Column('story_id', sa.Text), sa.Column('key', sa.Text),
    sa.Column('sector_stories', sa.Integer), sa.Column('payload', JSONB))
T_COMP = sa.Table('hot_company', meta, sa.Column('date', sa.Text, primary_key=True), sa.Column('company_code', sa.Text, primary_key=True),
    sa.Column('created_kst', sa.Text), sa.Column('company', sa.Text), sa.Column('impact', sa.Text), sa.Column('line', sa.Text),
    sa.Column('story_id', sa.Text), sa.Column('key', sa.Text), sa.Column('payload', JSONB))
T_KO = sa.Table('article_ko', meta, sa.Column('key', sa.Text, primary_key=True), sa.Column('text', sa.Text), sa.Column('impact_note', sa.Text),
    sa.Column('created_kst', sa.Text))   # 뉴스 상세 한국어 번역(news_ko_summary)
T_CSUM = sa.Table('company_summary', meta, sa.Column('date', sa.Text, primary_key=True), sa.Column('company_code', sa.Text, primary_key=True),
    sa.Column('created_kst', sa.Text), sa.Column('company', sa.Text), sa.Column('summary', sa.Text),
    sa.Column('company_stories', sa.Integer), sa.Column('company_articles', sa.Integer), sa.Column('payload', JSONB))
T_REL = sa.Table('company_relation', meta, sa.Column('id', sa.Text, primary_key=True),   # 종목코드|상대 노드
    sa.Column('stock_code', sa.Text), sa.Column('company', sa.Text), sa.Column('node', sa.Text), sa.Column('partner', sa.Text),
    sa.Column('line', sa.Text), sa.Column('rel_hash', sa.Text), sa.Column('created_kst', sa.Text), sa.Column('model', sa.Text))
T_REASON = sa.Table('company_reason', meta, sa.Column('date', sa.Text, primary_key=True), sa.Column('company_code', sa.Text, primary_key=True),
    sa.Column('created_kst', sa.Text), sa.Column('company', sa.Text), sa.Column('route', sa.Text), sa.Column('entity_name', sa.Text),
    sa.Column('partner', sa.Text), sa.Column('partner_node', sa.Text), sa.Column('rule', sa.Text), sa.Column('impact', sa.Text), sa.Column('stock_impact', sa.Text),
    sa.Column('direction', sa.Text), sa.Column('pick', sa.Text), sa.Column('key', sa.Text), sa.Column('story_id', sa.Text),
    sa.Column('line', sa.Text), sa.Column('relation_line', sa.Text), sa.Column('payload', JSONB))
T_SIMP = sa.Table('stock_impacts', meta, sa.Column('key', sa.Text, primary_key=True), sa.Column('stock_code', sa.Text, primary_key=True),
    sa.Column('impact', sa.Text), sa.Column('route', sa.Text), sa.Column('partner', sa.Text), sa.Column('rule', sa.Text), sa.Column('computed_kst', sa.Text))   # 2026-11-22
meta.create_all(engine)
with engine.begin() as con:   # 2026-11-20 뒤에 더한 칸(이미 있는 표에)
    for col, typ in (('title', 'text'), ('headings', 'jsonb'), ('keywords', 'jsonb')):
        con.exec_driver_sql(f'alter table news.hot_macro add column if not exists {col} {typ}')
    con.exec_driver_sql('alter table news.stories add column if not exists news_id text')   # 2026-11-22
    con.exec_driver_sql('alter table news.llm_articles add column if not exists original_url text')   # 언론사 원문 주소(2026-11-22)
    con.exec_driver_sql('alter table news.article_ko add column if not exists text text')   # 줄글 번역(2026-11-20, news_ko_summary)


def upsert(table, rows, chunk=500):
    if not rows:
        return 0
    keys = [c.name for c in table.primary_key.columns]
    with engine.begin() as con:
        for i in range(0, len(rows), chunk):
            st = pg_insert(table).values(rows[i:i + chunk])
            con.execute(st.on_conflict_do_update(index_elements=keys, set_={c: st.excluded[c] for c in rows[0] if c not in keys}))
    return len(rows)

## 2. 파일 읽기
- 기사: 같은 기사가 여러 줄이면 마지막 줄(나중 처리)을 쓴다. 10분마다 돌므로 최근 `SYNC_DAYS`일 안에 처음 수집한 기사만 다시 올린다.
- 요약: 날짜(종목은 날짜·종목)마다 `created_kst`가 가장 늦은 것.

In [ ]:
def jsonl(name):
    p = LIVE / name
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()] if p.exists() else []


def latest(rows, key):
    out = {}
    for r in rows:
        k = key(r)
        if k not in out or str(r.get('created_kst', '')) >= str(out[k].get('created_kst', '')):
            out[k] = r
    return list(out.values())


def publisher_url(u):
    """news_extract가 구글 뉴스 링크를 풀어 얻은 언론사 원문 주소. 못 풀었으면(NOT_RESOLVED) 비워 두고 화면은 구글 뉴스 링크를 쓴다"""
    u = (u or '').strip()
    return u if u.startswith('http') and 'news.google.com' not in u else None


def article_rows():
    last = {r['key']: r for r in jsonl('articles.jsonl')}
    if SYNC_DAYS is not None:
        since = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=SYNC_DAYS)).isoformat()
        last = {k: r for k, r in last.items() if str(r.get('first_seen_utc') or '') >= since}
    rows = []
    for r in last.values():
        e = r.get('extraction') or {}
        prof, graph = e.get('article_profile') or {}, e.get('graph') or {}
        names = {x.get('id'): x for x in graph.get('entities') or []}
        impacts = [dict(name=(names.get(c.get('entity_id')) or {}).get('name'), name_en=(names.get(c.get('entity_id')) or {}).get('name_en'),
                        impact=c.get('impact')) for c in e.get('company_impact') or []]
        rows.append(dict(key=r['key'], report=r.get('report'), status=r.get('status'), short_summary=prof.get('short_summary'),
                         news_kind=prof.get('news_kind'), article_types=prof.get('article_type'), sectors=graph.get('sectors'),
                         impacts=impacts, extraction=e or None, extract_error=r.get('extract_error'), published_utc=r.get('published_utc'),
                         original_url=publisher_url(r.get('original_url'))))
    return rows


def csv_rows(name):
    p = LIVE / name
    return list(csv.DictReader(open(p, encoding='utf-8-sig'))) if p.exists() else []


to_int = lambda v: None if v in (None, '') else int(float(v))

## 2-1. 기사 × 내 종목 긍정·부정 (2026-11-22)

기사 추출 결과의 회사별 판정(`company_impact`)을 **이 기사가 걸린 우리 종목** 기준으로 바꾼다. 팀원 `news_company_reason.ipynb`와 같은 규칙이다. LLM은 부르지 않는다.
1. 기사가 어느 종목의 검색어로 걸렸는지: 수집 DB `hits` × `news.term_links`(검색 문자열 → 종목).
2. **자기 기사**: 기사 속 회사가 그 종목 자신(자기 회사 검색어 이름과 같음)이면 그 판정 그대로.
3. **관계사 기사**: 기사의 주인공(mention = 중심)이 그 종목의 해외 상대이고, 방향 규칙표 `relation_direction_rules.csv`가 `동일`이면 그대로, `반대`면 긍정 ↔ 부정. `불명`이거나 표에 없으면 판정하지 않는다.
4. 그 밖(제품·업종·테마로 걸린 기사)은 판정하지 않는다(줄을 만들지 않음).

판정이 생긴 줄만 `news.stock_impacts`(기사 · 종목 · 판정 · 경로 · 관계사 · 규칙 · 계산 시각)에 쌓는다. 다시 계산하는 기사(최근 `SYNC_DAYS`일)는 그 기사의 줄을 지우고 새로 넣으므로, 규칙표를 고치면 다음 실행에 반영된다. 지난 기사까지 다시 계산하려면 `SYNC_DAYS = None`으로 한 번 돌린다.

In [ ]:
import re, datetime as dt
from collections import defaultdict

RULES_PATH = ROOT / 'relation_direction_rules.csv'   # 팀원 방향 규칙표((종목, 관계사)마다 동일·반대·불명)
_SUFFIX = re.compile(r'\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|llc|holdings?|group|sa|ag|nv)\b')


def name_key(name):
    """기업 이름 비교용(news_company_reason.ipynb와 같음): 소문자, 앞의 the·끝의 Inc. 등·문장부호·공백을 없앤다."""
    s = (name or '').lower().replace('&', ' and ')
    s = re.sub(r'^the\s+', '', s)
    s = _SUFFIX.sub(' ', re.sub(r'[^\w\s]', ' ', s))
    return re.sub(r'\s+', '', s)


def stock_impact_rows(arts):
    """기사 목록(article_rows) → (다시 계산한 기사 key 목록, 판정 줄 목록)."""
    keys = [a['key'] for a in arts]
    if not keys:
        return [], []
    with engine.connect() as con:
        tl = con.execute(sa.text('select stock_code, kind, query, node, node_name, link from news.term_links')).fetchall()
        hits = con.execute(sa.text('select key, term from hits where key = any(:k)'), {'k': keys}).fetchall()
    rules = {(r['stock_code'], r['node']): r['rule'] for r in csv.DictReader(open(RULES_PATH, encoding='utf-8-sig'))} if RULES_PATH.exists() else {}
    by_query, own, partner = defaultdict(set), defaultdict(set), defaultdict(dict)
    for code, kind, q, node, node_name, link in tl:
        by_query[q].add(code)
        if kind == '자기 회사':
            own[code].add(name_key(q.replace('"', ' ')))
        rule = rules.get((code, node))
        if kind == '해외 상대' and link == 'direct' and rule in ('동일', '반대'):
            first = q.split('"')[1] if q.count('"') >= 2 else q
            for nm in (first, node_name, node.split(':', 1)[-1]):
                if name_key(nm):
                    partner[code][name_key(nm)] = (rule, node_name or node)
    terms_of = defaultdict(set)
    for k, t in hits:
        terms_of[k].add(t)
    now = dt.datetime.now(dt.timezone(dt.timedelta(hours=9))).strftime('%Y-%m-%d %H:%M:%S')
    out = []
    for a in arts:
        e = a.get('extraction') or {}
        ents = {x.get('id'): x for x in (e.get('graph') or {}).get('entities') or []}
        imp = [(ents.get(ci.get('entity_id')) or {}, ci.get('impact')) for ci in e.get('company_impact') or []]
        imp = [(x, i) for x, i in imp if i in ('positive', 'negative', 'neutral', 'mixed')]
        if not imp:
            continue
        for code in {c for t in terms_of.get(a['key'], ()) for c in by_query.get(t, ())}:
            row = None
            for x, i in imp:   # 자기 기사
                if {name_key(x.get('name_en')), name_key(x.get('name'))} & own[code] - {''}:
                    row = dict(impact=i, route='자기', partner=None, rule=None)
                    break
            if row is None:
                for x, i in imp:   # 관계사 기사(주인공일 때만)
                    if x.get('mention') != '중심':
                        continue
                    hit = next((partner[code][k] for k in (name_key(x.get('name_en')), name_key(x.get('name'))) if k and k in partner[code]), None)
                    if hit:
                        rule, pname = hit
                        v = i if rule == '동일' or i in ('neutral', 'mixed') else ('negative' if i == 'positive' else 'positive')
                        row = dict(impact=v, route='관계사', partner=pname, rule=rule)
                        break
            if row:
                out.append(dict(key=a['key'], stock_code=code, computed_kst=now, **row))
    return keys, out


def save_stock_impacts(arts):
    keys, rows = stock_impact_rows(arts)
    with engine.begin() as con:   # 다시 계산한 기사의 줄을 지우고 새로 넣는다(규칙이 바뀌어 판정이 없어진 줄도 지워진다)
        for i in range(0, len(keys), 2000):
            con.execute(sa.text('delete from news.stock_impacts where key = any(:k)'), {'k': keys[i:i + 2000]})
    return upsert(T_SIMP, rows)

## 3. 올리기

In [ ]:
def sync():
    done = {}
    arts = article_rows()
    done['llm_articles'] = upsert(T_ART, arts)
    done['stock_impacts'] = save_stock_impacts(arts)
    news_of = {r['key']: r.get('news_id') for r in jsonl('llm_source.jsonl') if r.get('key')}   # 같은 기사가 여러 줄이면 마지막 줄
    done['stories'] = upsert(T_STORY, [dict(key=r['key'], story_id=r['story_id'], joined_by=r.get('joined_by'), news_id=news_of.get(r['key']) or None)
                                       for r in csv_rows('stories.csv')])
    done['importance'] = upsert(T_IMP, [dict(story_id=r['story_id'], news_id=r.get('news_id') or '', short_summary=r.get('short_summary'),
                                             news_outlets=to_int(r.get('news_outlets')), story_outlets=to_int(r.get('story_outlets')),
                                             story_articles=to_int(r.get('story_articles')), first_published=r.get('first_published'),
                                             story_last_published=r.get('story_last_published')) for r in csv_rows('importance.csv')])
    heads = {h['date']: h for h in latest(jsonl('macro_headlines.jsonl'), lambda r: r['date'])}
    done['hot_macro'] = upsert(T_MACRO, [dict(date=r['date'], created_kst=r.get('created_kst'), lines=r.get('lines'), keys=r.get('keys'), payload=r,
                                              title=(heads.get(r['date']) or {}).get('title'), headings=(heads.get(r['date']) or {}).get('headings'),
                                              keywords=(heads.get(r['date']) or {}).get('keywords'))
                                         for r in latest(jsonl('macro_hot_issues.jsonl'), lambda r: r['date'])])
    sec = []
    for r in latest(jsonl('sector_hot_issues.jsonl'), lambda r: r['date']):
        for s in r.get('sector_issues') or []:
            sec.append(dict(date=r['date'], sector=s['sector'], created_kst=r.get('created_kst'), line=s.get('line'), story_id=s.get('story_id'),
                            key=s.get('key'), sector_stories=s.get('sector_stories'), payload=s))
    done['hot_sector'] = upsert(T_SECTOR, sec)
    done['hot_company'] = upsert(T_COMP, [dict(date=r['date'], company_code=r['company_code'], created_kst=r.get('created_kst'), company=r.get('company'),
                                               impact=r.get('impact'), line=r.get('line'), story_id=r.get('story_id'), key=r.get('key'), payload=r)
                                          for r in latest(jsonl('company_hot_issues.jsonl'), lambda r: (r['date'], r['company_code']))])
    done['company_summary'] = upsert(T_CSUM, [dict(date=r['date'], company_code=r['company_code'], created_kst=r.get('created_kst'), company=r.get('company'),
                                                   summary=r.get('summary'), company_stories=r.get('company_stories'), company_articles=r.get('company_articles'),
                                                   payload=r) for r in latest(jsonl('company_summaries.jsonl'), lambda r: (r['date'], r['company_code']))])
    done['company_relation'] = upsert(T_REL, [dict(id=r['id'], stock_code=r.get('stock_code'), company=r.get('company'), node=r.get('node'), partner=r.get('partner'),
                                                  line=r.get('line'), rel_hash=r.get('rel_hash'), created_kst=r.get('created_kst'), model=r.get('model'))
                                             for r in latest(jsonl('company_relations.jsonl'), lambda r: r['id'])])
    done['company_reason'] = upsert(T_REASON, [dict(date=r['date'], company_code=r['company_code'], created_kst=r.get('created_kst'), company=r.get('company'),
                                                    route=r.get('route'), entity_name=r.get('entity_name'), partner=r.get('partner'), partner_node=r.get('partner_node'),
                                                    rule=r.get('rule'), impact=r.get('impact'), stock_impact=r.get('stock_impact'), direction=r.get('direction'),
                                                    pick=r.get('pick'), key=r.get('key'), story_id=r.get('story_id'), line=r.get('line'),
                                                    relation_line=r.get('relation_line'), payload=r)
                                               for r in latest(jsonl('company_reasons.jsonl'), lambda r: (r['date'], r['company_code']))])
    ko = list({r['key']: r for r in jsonl('ko_summaries.jsonl') if r.get('text')}.values())   # 번역(text)이 있는 줄만, 같은 기사는 마지막 줄
    done['article_ko'] = upsert(T_KO, [dict(key=r['key'], text=r['text'], impact_note=r.get('impact_note'), created_kst=r.get('created_kst')) for r in ko])
    print('DB에 올림', done)
    return done

## 4. 실행
서버에서는 `run_news.py`가 요약 뒤와 10분마다 부른다. 처음 한 번 전부 올릴 때는 `SYNC_DAYS = None`으로.

In [ ]:
sync()